In [ ]:
from openai import OpenAI
import os, json, base64, mimetypes, time, sys
from pathlib import Path
from typing import Dict, Any, List
import pandas as pd
from tqdm import tqdm
from PIL import Image

# ---------------- CONFIG ----------------

# Safer: ask for key inside notebook (won't save inside .ipynb)
OPENAI_API_KEY = input("Enter your OpenAI API key: ").strip()

if not OPENAI_API_KEY:
    raise RuntimeError("OpenAI key is empty!")

client = OpenAI(api_key=OPENAI_API_KEY)

# AVAILABLE MODELS:
# "gpt-4o"          → Full 4o
# "gpt-4o-mini"     → Cheaper, fast baseline
# "gpt-5"           → Latest GPT-5 model
# "gpt-5-mini"      → Cheaper GPT-5 lightweight

MODEL = "o4-mini"    # <-- CHANGE THIS as needed

INPUT_PATH  = "../MedGemma/test_flat.jsonl"
OUTPUT_CSV  = f"{MODEL.replace('.', '_')}_predictions.csv"

MAX_TOKENS       = 512
TEMPERATURE      = 0.0
MAX_RETRIES      = 5
BACKOFF_BASE     = 2.0
BATCH_SAVE_EVERY = 25


In [ ]:
# ---------------- HELPERS ----------------

def encode_image_to_base64(path: str):
    """Load image file → base64 encode → return (mime, b64)."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")

    mime, _ = mimetypes.guess_type(str(path))
    if mime is None:
        mime = "image/png"

    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    return mime, b64


def ask_gpt_vqa(image_path: str, question: str) -> str:
    """Send image + question to GPT-4o / GPT-5 / GPT-5-mini."""

    # Load and convert image to data URL
    mime, b64 = encode_image_to_base64(image_path)
    img_url = f"data:{mime};base64,{b64}"

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            completion = client.chat.completions.create(
                model=MODEL,

                # IMPORTANT: GPT-4o/GPT-5 require this (not max_tokens)
                max_completion_tokens=MAX_TOKENS,

                # IMPORTANT: GPT-4o/GPT-5 DO NOT allow temperature control anymore
                # temperature=1 is the implicit default (so we remove it)

                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a medical VQA assistant specializing in spine X-rays. "
                            "Answer concisely, clinically, and in one sentence. "
                            "If unsure, say 'I am unsure based on this image.'"
                        )
                    },
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image_url",
                                "image_url": { "url": img_url }
                            },
                            {
                                "type": "text",
                                "text": (
                                    f"Question: {question}\n"
                                    "Provide only the answer.\n"
                                    "Answer:"
                                )
                            }
                        ]
                    }
                ]
            )

            return completion.choices[0].message.content.strip()

        except Exception as e:
            print(f"[Attempt {attempt}] Error: {e}")
            time.sleep(BACKOFF_BASE ** (attempt - 1))

    return "ERROR: GPT request failed"



def load_jsonl(path: str):
    rows = []
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


In [ ]:
# ---------------- SANITY CHECK ----------------

def sanity_check(idx=0):
    data = load_jsonl(INPUT_PATH)
    print("Total samples:", len(data))

    sample = data[idx]
    img = sample["image_path"]
    q   = sample["question"]
    gt  = sample.get("answer", "")

    print("\n--- Sanity Check ---")
    print(f"Index: {idx}")
    print(f"Image path: {img}")
    print(f"Exists: {Path(img).exists()}")
    print(f"Question: {q}")
    print(f"Ground truth: {gt}")

    pred = ask_gpt_vqa(img, q)
    print("\nModel Prediction:", pred)


In [ ]:
sanity_check(0)


In [ ]:
# ---------------- MAIN LOOP (WITH RESUME) ----------------

def run_full_inference():
    data = load_jsonl(INPUT_PATH)
    print(f"Loaded {len(data)} samples.")

    rows = []
    start_idx = 0
    out_path = Path(OUTPUT_CSV)

    # Resume logic
    if out_path.exists():
        prev = pd.read_csv(out_path)
        rows = prev.to_dict(orient="records")
        start_idx = len(rows)
        print(f"Resuming from {start_idx} samples already processed.")

    for idx in tqdm(range(start_idx, len(data)), desc=f"{MODEL} VQA"):
        sample = data[idx]
        img = sample["image_path"]
        q   = sample["question"]
        gt  = sample.get("answer", "")

        try:
            pred = ask_gpt_vqa(img, q)
            time.sleep(0.25)
        except Exception as e:
            pred = f"ERROR: {e}"

        rows.append({
            "index": idx,
            "image_path": img,
            "question": q,
            "ground_truth": gt,
            "prediction": pred,
        })

        if (idx + 1) % BATCH_SAVE_EVERY == 0:
            pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
            print(f"Checkpoint saved at {idx+1}")

    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
    print(f"🎉 DONE! Saved {len(rows)} predictions → {OUTPUT_CSV}")


In [ ]:
run_full_inference()


### Evaluation

In [ ]:
# %% Install (uncomment & run once in your env)
# %pip install pandas nltk rouge-score bert-score

import re
from pathlib import Path

import pandas as pd
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score


# =========================
# CONFIG
# =========================
CSV_PATH = Path("./o4-mini_predictions.csv")

REF_COL = "ground_truth"             # ground-truth column name
PRED_COL = "prediction"  # prediction column name

# If some rows are clearly invalid predictions, you can filter by a prefix:
ERROR_PREFIX = "[ERROR]"       # set to None if you don't want this filter

# BERTScore config
BERT_MODEL_TYPE = "bert-base-uncased"  # you can switch to a larger model later
#BERT_LANG = "en"
BERT_BATCH_SIZE = 64                  # adjust based on your GPU memory
#BERT_RESCALE_BASELINE = True          # recommended for English


# =========================
# HELPERS
# =========================
def normalize_text(s: str) -> str:
    """Lowercase + strip + collapse spaces. Extend if you want more aggressive cleaning."""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def load_and_prepare(csv_path: Path, ref_col: str, pred_col: str, error_prefix: str | None = None):
    df = pd.read_csv(csv_path)

    # Drop rows with missing ref or pred
    df = df.dropna(subset=[ref_col, pred_col])

    # Optional: drop obvious error rows (e.g. "[ERROR] rate limited")
    if error_prefix:
        df = df[~df[pred_col].astype(str).str.startswith(error_prefix)]

    df = df.reset_index(drop=True)

    refs_raw = df[ref_col].astype(str).tolist()
    preds_raw = df[pred_col].astype(str).tolist()

    print(f"Using {len(refs_raw)} examples for evaluation")
    return refs_raw, preds_raw


def compute_accuracy(ref_norm, pred_norm):
    correct = sum(r == p for r, p in zip(ref_norm, pred_norm))
    total = len(ref_norm)
    acc = correct / total if total > 0 else 0.0
    return acc, correct, total


def compute_bleu(ref_norm, pred_norm):
    # NLTK expects list of list of tokenized references, and list of tokenized hypotheses
    refs_tok = [[r.split()] for r in ref_norm]   # one reference per example
    preds_tok = [p.split() for p in pred_norm]

    smooth = SmoothingFunction().method1
    bleu4 = corpus_bleu(
        refs_tok,
        preds_tok,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smooth,
    )
    return bleu4


def compute_rouge_l(ref_norm, pred_norm):
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    f_scores = []

    for r, p in zip(ref_norm, pred_norm):
        score = scorer.score(r, p)["rougeL"]
        f_scores.append(score.fmeasure)

    avg_f = sum(f_scores) / len(f_scores) if f_scores else 0.0
    return avg_f


def compute_bertscore(refs_raw, preds_raw):
    print("\nComputing BERTScore (this may take a bit)...")
    P, R, F1 = bertscore_score(
        cands=preds_raw,
        refs=refs_raw,
        model_type=BERT_MODEL_TYPE,
        #lang=BERT_LANG,
        batch_size=BERT_BATCH_SIZE,
        #rescale_with_baseline=BERT_RESCALE_BASELINE,
        verbose=True,
    )

    bert_p = float(P.mean())
    bert_r = float(R.mean())
    bert_f1 = float(F1.mean())
    return bert_p, bert_r, bert_f1


# =========================
# MAIN
# =========================
if __name__ == "__main__":
    # 1. Load data
    refs_raw, preds_raw = load_and_prepare(CSV_PATH, REF_COL, PRED_COL, ERROR_PREFIX)

    # 2. Normalize (for Accuracy, BLEU, ROUGE)
    ref_norm = [normalize_text(x) for x in refs_raw]
    pred_norm = [normalize_text(x) for x in preds_raw]

    # 3. Accuracy
    accuracy, correct, total = compute_accuracy(ref_norm, pred_norm)
    print(f"\nAccuracy (exact match, normalized): {accuracy:.4f}  ({correct}/{total})")

    # 4. BLEU-4
    bleu4 = compute_bleu(ref_norm, pred_norm)
    print(f"BLEU-4 (corpus, normalized): {bleu4:.4f}")

    # 5. ROUGE-L (F1)
    rouge_l_f1 = compute_rouge_l(ref_norm, pred_norm)
    print(f"ROUGE-L (F1, avg, normalized): {rouge_l_f1:.4f}")

    # 6. BERTScore (raw text)
    bert_p, bert_r, bert_f1 = compute_bertscore(refs_raw, preds_raw)
    print(f"BERTScore Precision (mean): {bert_p:.4f}")
    print(f"BERTScore Recall    (mean): {bert_r:.4f}")
    print(f"BERTScore F1        (mean): {bert_f1:.4f}")

    # 7. Summary dict (easy to log / save)
    metrics = {
        "accuracy_exact": accuracy,
        "bleu4": bleu4,
        "rougeL_f1": rouge_l_f1,
        "bertscore_p": bert_p,
        "bertscore_r": bert_r,
        "bertscore_f1": bert_f1,
    }

    print("\n=== Summary Metrics ===")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
